# index-by-tensor composite — cx20: mutate selected rows in place via index-by-tensor write

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `index-by-tensor`, `slice-view-mutation`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "index-by-tensor"
DD_ATOM_IDS = ["index-by-tensor", "slice-view-mutation"]
DD_SUBTOPICS = ["PyTorch: index by tensor", "PyTorch: Slice view mutation"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

`index-by-tensor` is normally a READ — `mat[idx]` returns gathered values. But PyTorch lets you also WRITE through that same expression: `mat[idx] = value` mutates the selected rows in place, sharing storage with the original tensor. That's the same underlying mechanic as slice-view-mutation (assigning to a view modifies the source) — just with a tensor index instead of a slice.

Composing these two atoms: select rows by an index tensor, write to them, and verify (a) the storage pointer didn't move, (b) the selected rows are now zero, and (c) the non-selected rows are untouched.

### Composite Exercise — mutate selected rows in place via index-by-tensor write

**Atoms exercised together**: `index-by-tensor`, `slice-view-mutation`

Implement `cx20_zero_rows_inplace(mat, idx)` that:

- Takes `mat` of shape `(N, D)` and `idx` of shape `(K,)` long (the row indices to zero).
- Zeros the selected rows IN PLACE via index-by-tensor write: `mat[idx] = 0.0` (or equivalent slice-view assignment).
- Returns `mat` (the same tensor object — must not reallocate).

The unchanged rows must be bit-identical, and `mat.data_ptr()` must not move.

In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

def cx20_zero_rows_inplace(mat, idx):
    raise NotImplementedError

def _test_cx20():
    mat = t.arange(20.0).reshape(5, 4) + 1.0
    # rows: [1,2,3,4],[5,6,7,8],[9,10,11,12],[13,14,15,16],[17,18,19,20]
    original_ptr = mat.data_ptr()
    before_others = mat[t.tensor([0, 2, 4])].clone()
    idx = t.tensor([1, 3], dtype=t.long)
    out = cx20_zero_rows_inplace(mat, idx)

    # Identity + storage preserved.
    assert out is mat, 'must return the same tensor object (in-place)'
    assert out.data_ptr() == original_ptr, 'must not reallocate'

    # Selected rows now zero.
    assert t.allclose(mat[1], t.zeros(4)), f'row 1 not zeroed: {mat[1]}'
    assert t.allclose(mat[3], t.zeros(4)), f'row 3 not zeroed: {mat[3]}'

    # Non-selected rows untouched.
    after_others = mat[t.tensor([0, 2, 4])]
    assert t.allclose(before_others, after_others), 'non-selected rows mutated'

    # Larger random case.
    rng = t.Generator().manual_seed(11)
    big = t.randn(20, 8, generator=rng)
    snap = big.clone()
    kill = t.tensor([0, 5, 7, 19], dtype=t.long)
    keep_mask = t.ones(20, dtype=t.bool); keep_mask[kill] = False
    cx20_zero_rows_inplace(big, kill)
    assert t.allclose(big[kill], t.zeros(4, 8))
    assert t.allclose(big[keep_mask], snap[keep_mask]), 'kept rows must be untouched'

    # Empty idx is a no-op.
    small = t.ones(3, 2)
    ptr = small.data_ptr()
    cx20_zero_rows_inplace(small, t.tensor([], dtype=t.long))
    assert t.allclose(small, t.ones(3, 2)), 'empty idx must leave mat untouched'
    assert small.data_ptr() == ptr
    _dd_passed.add('cx20')

_test_cx20()

<details><summary>Show solution — cx20</summary>

```python
def cx20_zero_rows_inplace(mat, idx):
    # Index-by-tensor on the LHS: PyTorch resolves mat[idx] to a slice-view of the
    # selected rows, then broadcasts the RHS scalar zero across them. Storage is
    # shared with mat — this is the same mechanic as slice-view mutation.
    mat[idx] = 0.0
    return mat
```

Index-by-tensor works on both sides of `=`. On the right it READS (gather); on the left it WRITES (scatter). The write path shares storage with `mat` — there's no copy and no reallocation, which is why `data_ptr()` is unchanged. Don't reach for `index_copy_` or `scatter_` unless you need their specific semantics — plain `mat[idx] = value` is the idiom for this composition.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx20'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx20',
        'subtopics': ["PyTorch: index by tensor", "PyTorch: Slice view mutation"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()